In [1]:
import numpy as np
import pandas as pd
import itertools
import matplotlib
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import date
import re
# from tqdm import tqdm
import time
# from pybedtools import BedTool
import random
import math
import pickle
import seaborn as sns
from scipy.stats import percentileofscore, pearsonr, rankdata
# from scipy.spatial.distance import jaccard, euclidean, cosine
# import functools
# import operator
import gzip
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import sys
import os
from functools import partial
from contextlib import closing
from multiprocessing import Pool
import re
import h5py
import tabix
import upsetplot

today = date.today()

**Curate TF dataset**

In [29]:
#loading TF motif names
f = open('/home/ctrader/region_similarity/data/regions_25kb_CT20220613.TFmapped.bed', 'r')
TF_component_order = np.array(f.readline().strip().split('\t'))[3:]
f.close()

In [64]:
gene_dataset = pd.read_csv('/home/ctrader/region_similarity/data/mart_export_protein_coding_genes_CT20220624.txt', delimiter='\t', header=0)
gene_dataset = gene_dataset[gene_dataset['Gene name'].notna()]

In [65]:
#using gene descriptors for tf motifs
TFfact2descr = {}
for name in TF_component_order:
    name_split = name.split('_')
    if len(name_split)>1:
        keep_num = False
    else:
        keep_num = True
    root = name_split[0]
    
    descriptors = np.array(gene_dataset[gene_dataset['Gene name'].str.contains(root, case=False)]['Gene description'])
    if len(descriptors) > 0:
        descriptor = descriptors[0].split(' [')[0]
        if ((not keep_num) and (not descriptor[-1].isalpha())):
            descriptor = " ".join(descriptor.split(' ')[:-1])
        TFfact2descr[name]=descriptor
    else:
        TFfact2descr[name]=np.nan

In [66]:
#manual entries
TFfact2descr['ZNF435'] = 'zinc finger protein 435'
TFfact2descr['ZNF332'] = 'zinc finger protein 332'
TFfact2descr['ZNF306'] = 'zinc finger protein 306'
TFfact2descr['ZFN121'] = 'zinc finger protein 121'
TFfact2descr['HINFP1_1'] = 'Histone H4 Transcription Factor'
TFfact2descr['HINFP1_2'] = 'Histone H4 Transcription Factor'
TFfact2descr['HINFP1_3'] = 'Histone H4 Transcription Factor'
TFfact2descr['ARI5A'] = 'AT-Rich Interaction Domain'
TFfact2descr['ARI5B'] = 'AT-Rich Interaction Domain'
TFfact2descr['CCAAT_CEBP'] = 'Enhancer Binding Protein'
TFfact2descr['CENBP'] = 'Centromere Protein B'
TFfact2descr['DDIT3+CEBPA'] = 'DNA Damage Inducible Transcript + CCAAT Enhancer Binding Protein Alpha'
TFfact2descr['EVI1_MECOM'] = 'MDS1 And EVI1 Complex Locus'
TFfact2descr['GC-tract'] = 'GC Vitamin D Binding Protein'
TFfact2descr['HEN1'] = 'Small RNA 2\'-O-methyltransferase'
TFfact2descr['KAISO'] = 'Zinc Finger And BTB Domain Containing 33'
TFfact2descr['MFZ1'] = 'C2H2'
TFfact2descr['MIES'] = 'Meis Homeobox'
TFfact2descr['NFAC_2'] = 'Nuclear Factor Of Activated T Cells'
TFfact2descr['OCT4+SOX2'] = 'Octamer-Binding Protein + SRY-Box Transcription Factor'
TFfact2descr['P53-like_1'] = 'Tumor Protein P53'
TFfact2descr['P53-like_2'] = 'Tumor Protein P53'
TFfact2descr['P53-like_3'] = 'Tumor Protein P53'
TFfact2descr['PAX-halfsite'] = 'Paired Box Protein'
TFfact2descr['REL-halfsite'] = 'Proto-Oncogene, NF-KB Subunit'
TFfact2descr['TATA'] = 'TATA-Box Binding Protein'

for name, descriptor in zip(list(TFfact2descr.keys()), list(TFfact2descr.values())):
    if not isinstance(descriptor, str):
        print(name, descriptor)

In [67]:
pickle_out = open('/home/ctrader/Hold/NMF/DHS/pickled_data/tf_motif_descriptor_dict_CT20220627.pickle', 'wb')
pickle.dump(TFfact2descr, pickle_out)
pickle_out.close()

**DHS descriptors**

In [6]:
f = open('/home/ctrader/region_similarity/data/regions_25kb_CT20220708.DHSmapped.bed', 'r')
DHS_component_order = np.array(f.readline().strip().split('\t'))[3:]
f.close()

In [7]:
DHSfact2descr = {}
for factor in DHS_component_order:
    DHSfact2descr[factor] = 'DNase I accessibility in ' + '/'.join(factor.split('_')) + ' samples'

In [8]:
pickle_out = open('/home/ctrader/region_similarity/data/DHS_descriptor_dict_CT20220729.pickle', 'wb')
pickle.dump(DHSfact2descr, pickle_out)
pickle_out.close()

**Epilogos descriptors**

In [61]:
epilogosfact2descr = {}
f = open('/home/stsai/AugmentedGenomics/externalresources/data/All_833_biosamples/metadata.tsv', 'r')
next(f)
for line in f.readlines():
    line = line.strip().split('\t')
    epilogosfact2descr['_'.join(line[2].split('/'))] = line[3]
f.close()

In [63]:
pickle_out = open('/home/ctrader/Hold/NMF/DHS/pickled_data/epilogos_descriptor_dict_CT20220627.pickle', 'wb')
pickle.dump(epilogosfact2descr, pickle_out)
pickle_out.close()